# 01 — EDA & Giảm chiều (PCA/t-SNE)

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


In [ ]:
# === Thiết lập chung (Kaggle) ===
import sys, os
# Cách 1: clone repo (khuyến nghị để đồng bộ code):
# !git clone https://github.com/ducqwang/FallDetection.git /kaggle/working/FD
# REPO='/kaggle/working/FD'
# Cách 2: notebook nằm sẵn trong repo:
REPO = '..' if os.path.exists('../src') else '/kaggle/working/FD'
sys.path.insert(0, REPO)
from src.utils import set_seed, load_config, on_kaggle
cfg = load_config(os.path.join(REPO, 'configs/default.yaml'))
set_seed(cfg['seed'])
# TRÊN KAGGLE: sửa cfg['paths'] trỏ tới /kaggle/input/<dataset-slug>/...
# cfg['paths']['urfd_root'] = '/kaggle/input/urfd/...'
# cfg['paths']['le2i_root'] = '/kaggle/input/le2i/...'
print('Kaggle:', on_kaggle())


## 1. Index dữ liệu + chia tập theo video (cross-dataset)


In [ ]:
from src.data_splits import build_splits
splits = build_splits(cfg)   # tạo splits/splits.json (within + cross_A/B)
items = splits['_items']
print('Tổng video:', len(items))
import collections
print(collections.Counter((it['dataset'], it['label']) for it in items))


## 2. Bốn biểu đồ EDA phân bố


In [ ]:
import matplotlib.pyplot as plt, collections
fig, ax = plt.subplots(1, 2, figsize=(11,4))
c = collections.Counter((it['dataset'], 'fall' if it['label'] else 'adl') for it in items)
labels=[f'{d}-{l}' for (d,l) in c]; ax[0].bar(labels, list(c.values())); ax[0].set_title('Số video theo lớp/nguồn'); ax[0].tick_params(axis='x', rotation=30)
scenes = collections.Counter(it.get('scene') for it in items)
ax[1].bar([str(s) for s in scenes], list(scenes.values())); ax[1].set_title('Phân bố góc/bối cảnh'); ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('outputs/eda_dist.png', dpi=120); plt.show()


## 3. Biểu đồ 5 — giảm chiều đặc trưng (BẮT BUỘC §2.2)
Trích đặc trưng hình học/pose cho từng video test rồi PCA/t-SNE tô màu fall/non-fall.


In [ ]:
import numpy as np
from src.detection_tracking import DetectorTracker
from src.datasets import make_gt_falls, geometric_windows, video_fps
from src.features import reduce_and_plot
dt = DetectorTracker(cfg['detection']['model'], cfg['detection']['conf'])
Xall, yall = [], []
for it in items[:40]:              # lấy mẫu để nhanh; bỏ [:40] khi chạy đầy đủ
    tracks = dt.run(it['path'])
    n = max((t.frame for t in tracks), default=0)+1
    X, y = geometric_windows(tracks, make_gt_falls(it, n), cfg['data']['seq_len'])
    if len(X): Xall.append(X); yall.append(y)
X = np.concatenate(Xall); y = np.concatenate(yall)
reduce_and_plot(X, y, method='tsne', out_png='outputs/tsne.png')
print('t-SNE lưu ở outputs/tsne.png — đưa vào báo cáo')
